# 02 — CNN Baseline (PyTorch, From Scratch)
**Objectif** : Baseline avec un CNN 4 blocs entraîné from scratch en PyTorch.
On identifie le surapprentissage sans transfer learning.


In [1]:
import sys; sys.path.append('../../src')
import torch, torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from data_loader import get_dataloaders, compute_class_weights
from models import CNNBaseline
from utils import plot_learning_curves, plot_confusion_matrix
import warnings; warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
torch.manual_seed(42)


Device : cpu


In [2]:
# ── Chargement des données ────────────────────────────────────
train_loader, val_loader, test_loader, train_ds, _, _ = get_dataloaders(
    '../../data/chest_xray/train',
    '../../data/chest_xray/val',
    '../../data/chest_xray/test',
    augment=False, batch_size=32)   # sans augmentation pour isoler l'architecture

cw = compute_class_weights(train_ds, DEVICE)
print(f"Poids de classe : {cw}")


Classes : {'NORMAL': 0, 'PNEUMONIA': 1}
Train   : 5216 | Val: 16 | Test: 624
Poids de classe : tensor([1.9448, 0.6730])


In [3]:
# ── Architecture CNN Baseline ─────────────────────────────────
#
#  Input (3×224×224)
#  Conv(32)→BN→ReLU→MaxPool(2) × 4 blocs
#  AdaptiveAvgPool(4×4)
#  Flatten → Linear(512) → ReLU → Dropout(0.5)
#  Linear(3)   [CrossEntropyLoss = log_softmax intégré]
#
model = CNNBaseline().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nTotal paramètres : {total_params:,}")


CNNBaseline(
  (features): Sequential(
    (0): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (2): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padd

In [4]:
# ── Entraînement ─────────────────────────────────────────────
criterion = nn.CrossEntropyLoss(weight=cw)
optimizer = Adam(model.parameters(), lr=1e-3)
scheduler = ReduceLROnPlateau(optimizer, factor=0.5, patience=3, verbose=True)

history    = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
best_acc   = 0.
patience   = 5
no_improve = 0

for epoch in range(1, 31):
    # ── Train ──
    model.train()
    tr_loss, tr_correct, tr_total = 0., 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward(); optimizer.step()
        tr_loss    += loss.item() * imgs.size(0)
        tr_correct += (out.argmax(1) == labels).sum().item()
        tr_total   += imgs.size(0)

    # ── Val ──
    model.eval()
    vl_loss, vl_correct, vl_total = 0., 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out  = model(imgs)
            loss = criterion(out, labels)
            vl_loss    += loss.item() * imgs.size(0)
            vl_correct += (out.argmax(1) == labels).sum().item()
            vl_total   += imgs.size(0)

    tr_acc = tr_correct / tr_total
    vl_acc = vl_correct / vl_total
    scheduler.step(vl_loss / vl_total)

    history['train_loss'].append(tr_loss / tr_total)
    history['val_loss'].append(vl_loss / vl_total)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    print(f"Ep {epoch:2d}  train_acc={tr_acc:.4f}  val_acc={vl_acc:.4f}")

    if vl_acc > best_acc:
        best_acc   = vl_acc
        no_improve = 0
        torch.save(model.state_dict(), '../../models/cnn_baseline.pt')
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"EarlyStopping déclenché à l'époque {epoch}"); break


TypeError: ReduceLROnPlateau.__init__() got an unexpected keyword argument 'verbose'

In [ ]:
# ── Courbes d'apprentissage ───────────────────────────────────
plot_learning_curves(history, 'CNN Baseline')
print(f"Best val_accuracy : {best_acc:.4f}")
print(f"Gap train/val     : {max(history['train_acc']) - best_acc:.4f}")


In [ ]:
# ── Évaluation test ───────────────────────────────────────────
CLASSES = ['NORMAL', 'BACTERIA', 'VIRUS']
model.load_state_dict(torch.load('../../models/cnn_baseline.pt', map_location=DEVICE))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)
        out  = torch.softmax(model(imgs), dim=1).cpu().numpy()
        all_probs.extend(out)
        all_preds.extend(out.argmax(1).tolist())
        all_labels.extend(labels.numpy().tolist())

print(classification_report(all_labels, all_preds, target_names=CLASSES))
plot_confusion_matrix(all_labels, all_preds)
print("OBSERVATION: ~87% accuracy — surapprentissage visible → transfer learning nécessaire")
